# InspectEscrow PATCH3 — Colab Frozen Input Integrity Gate

**Objective:** Verify bytes of the original PatchCore V2 model checkpoint, frozen memory bank, and exact raw MVTec image *before* any claimed fresh replay.

This notebook does **not** execute PatchCore inference, compute a new replay score, sign an Inspector attestation, select a settlement outcome or send transactions.

## Source location verified via connected Google Drive

- Model: `MeTal_Dataset/InspectEscrow_AI_MODEL_V2/patchcore_v2_model_core.pt` (232,188,217 bytes)
- Memory bank: `MeTal_Dataset/InspectEscrow_AI_MODEL_V2/MODEL_BUNDLE/patchcore_v2_memory_bank.pt` (132,466,400 bytes)
- Original image: `demo/scratch/022.png` inside `MeTal_Dataset/InspectEscrow_metal_nut_FINAL_DATASET.zip`.

Expected digests below originate from prior frozen integrity manifests. Current session must recompute them from *actual bytes*. This validates file identity, **not** actual inference execution or same-physical-specimen claims.

**DO NOT** paste private keys or wallet secrets into Google Colab. Replay model artifacts must be trusted before any use of PyTorch deserialization. The optional checkpoint preview uses only `torch.load(..., weights_only=True)`; if it fails, it does not automatically allow unsafe pickle.

## Trade binding

- Trade: `0x8e07587390de665a8a3c0fe81e1e7298541847acd244db5ea98e5dc3acb3f9db`
- PATCH3 escrow: `0x0a4721CF41Ad3f7a9EC9C73049D2b5D2db73B905`
- Buyer EvidencePackageHash: `0xcbeb27a5529a4dff9d9efc840708349769c8c79ab71632ccac3d3244e9d5f2b9`
- Source image `sha256`: `f88d2d084303ec537991d3f5263c4a9ba50eff042e4968ff8be2aebfe59e456a`
- Model core `sha256`: `8ab4beeb37242a742bac75c8f697d4af625eaf5acda339c887d6c77c9f6bd36b`
- Memory bank `keccak256`: `0x4761c52147167bcfb846dbf2362ab72dc94badb388bf054ec5e65ca609985b06`

The old 2026-10-10 Colab notebook `CELL 10` wraps historical `DISPUTE_022_REPLAY_RESULT.json`; re-running it is NOT a fresh inference execution.

In [ ]:
from pathlib import Path
import hashlib, io, json, zipfile

from google.colab import drive
drive.mount('/content/drive')

# Trusted project paths. No uploaded files, no wallet keys.
ROOT = Path('/content/drive/MyDrive/MeTal_Dataset')
V2 = ROOT / 'InspectEscrow_AI_MODEL_V2'
MODEL = V2 / 'patchcore_v2_model_core.pt'
BANK = V2 / 'MODEL_BUNDLE' / 'patchcore_v2_memory_bank.pt'
DATA = ROOT / 'InspectEscrow_metal_nut_FINAL_DATASET.zip'

EXPECTED_MODEL_SIZE = 232_188_217
EXPECTED_BANK_SIZE = 132_466_400
EXPECTED_MODEL_SHA256 = '8ab4beeb37242a742bac75c8f697d4af625eaf5acda339c887d6c77c9f6bd36b'
EXPECTED_BANK_KECCAK256 = '4761c52147167bcfb846dbf2362ab72dc94badb388bf054ec5e65ca609985b06'
EXPECTED_IMAGE_SHA256 = 'f88d2d084303ec537991d3f5263c4a9ba50eff042e4968ff8be2aebfe59e456a'

for name,p in [('model_core',MODEL),('memory_bank',BANK),('dataset_zip',DATA)]:
    assert p.is_file(), f'MISSING {name}: {p}'
assert MODEL.stat().st_size == EXPECTED_MODEL_SIZE, 'Model size mismatch'
assert BANK.stat().st_size == EXPECTED_BANK_SIZE, 'Memory bank size mismatch'
print('Three original artifacts found and expected file sizes verified.')


In [ ]:
# Cryptographic verification of original model / memory bank byte streams.
try:
    from Crypto.Hash import keccak
except ImportError as e:
    raise RuntimeError(
        'PyCryptodome is missing. In a separate cell run: %pip install pycryptodome'
    ) from e

def hash_file(p, hash_obj):
    with p.open('rb') as f:
        while True:
            buf=f.read(8*1024*1024)
            if not buf: break
            hash_obj.update(buf)
    return hash_obj.hexdigest()

model_sha = hash_file(MODEL, hashlib.sha256())
bank_keccak = hash_file(BANK, keccak.new(digest_bits=256))
assert model_sha == EXPECTED_MODEL_SHA256, 'STOP: model core file does not match frozen checkpoint!'
assert bank_keccak == EXPECTED_BANK_KECCAK256, 'STOP: memory bank does not match frozen artifact!'
print('PASS: exact original model core SHA256 and memory bank Keccak256.')


In [ ]:
# Validate the actual source image BYTES from the original dataset ZIP.
with zipfile.ZipFile(DATA,'r') as z:
    names=[n for n in z.namelist() if n.replace('\\','/').endswith('/demo/scratch/022.png')
           or n.replace('\\','/')=='demo/scratch/022.png']
    assert len(names)==1, f'STOP: expected exactly one scratch/022.png, got {names}'
    member=names[0]
    image_bytes=z.read(member)
image_sha=hashlib.sha256(image_bytes).hexdigest()
assert image_sha == EXPECTED_IMAGE_SHA256, 'STOP: original MVTec source image hash mismatch!'
from PIL import Image
with Image.open(io.BytesIO(image_bytes)) as im:
    assert im.format=='PNG' and im.mode=='RGB' and im.size==(700,700), (
        'Unexpected image format/mode/size')
print('PASS: exact dataset image bytes, SHA256 and 700x700 RGB PNG.')
print('Dataset member:',member)
print('Image SHA256:',image_sha)


In [ ]:
# Save a PRE-INFERENCE integrity receipt to the user's own Drive folder.
# It is NOT a new replay report, and cannot be used as ReplayAttestation.
import datetime
receipt={
    'schema':'inspectescrow.patch3.frozen-source-inputs-preflight/1',
    'tradeId':'0x8e07587390de665a8a3c0fe81e1e7298541847acd244db5ea98e5dc3acb3f9db',
    'modelBundleHash':'0x64f3359a4e9a276a5f0abef0e5305f42b513dbae1ac5698cb08c9e066962383e',
    'expectedBuyerEvidencePackageHash':'0xcbeb27a5529a4dff9d9efc840708349769c8c79ab71632ccac3d3244e9d5f2b9',
    'modelCorePath':str(MODEL),
    'modelCoreSha256':'0x'+model_sha,
    'memoryBankPath':str(BANK),
    'memoryBankKeccak256':'0x'+bank_keccak,
    'datasetPath':str(DATA),
    'datasetMember':member,
    'originalImageSha256':'0x'+image_sha,
    'frozenByteIntegrityPassed':True,
    'observedAtUtc':datetime.datetime.now(datetime.timezone.utc).isoformat(),
    'freshReplayPerformed':False,
    'rawReplayScore':None,
    'replayReportHash':None,
    'inspectorHumanDecision':None,
    'onchainTransactionsSent':0,
    'samePhysicalSpecimenProven':False
}
out = V2 / 'PATCH3_SOURCE_BYTES_PREFLIGHT_20261027.json'
out.write_text(json.dumps(receipt,indent=2,ensure_ascii=False)+'\n',encoding='utf-8')
print(json.dumps({
  'status':'PASS_PATCH3_COLAB_FROZEN_BINARY_INTEGRITY',
  'tradeId':receipt['tradeId'],
  'modelCoreSha256':receipt['modelCoreSha256'],
  'memoryBankKeccak256':receipt['memoryBankKeccak256'],
  'originalImageSha256':receipt['originalImageSha256'],
  'receiptPath':str(out),
  'freshReplayPerformed':False,
  'replayReportHash':None,
  'transactionsSent':0,
  'activateCalled':False
},indent=2))


In [ ]:
# OPTIONAL trusted-checkpoint structure preview (NOT inference).
# Do not run this if you do not trust the original Drive checkpoint.
import torch
try:
    obj=torch.load(MODEL,map_location='cpu',weights_only=True)
except Exception as e:
    print('SAFE_LOAD_UNSUPPORTED; no unsafe pickle fallback performed.')
    print('Exception class:', type(e).__name__)
    print('Use the original trusted model saving/inference code in a separate audited runtime.')
else:
    def shape_summary(value):
        if isinstance(value,torch.Tensor): return {'type':'Tensor','shape':list(value.shape),'dtype':str(value.dtype)}
        if isinstance(value,dict): return {'type':'dict','keys':list(map(str,list(value.keys())[:40]))}
        if isinstance(value,(list,tuple)): return {'type':type(value).__name__,'length':len(value)}
        return {'type':type(value).__name__}
    print('SAFE_LOAD_SUPPORTED:', shape_summary(obj))
    if isinstance(obj,dict):
        print('CHECKPOINT_COMPONENTS:')
        for k,v in list(obj.items())[:25]:
            print(str(k), shape_summary(v))
    del obj
